In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/birdclef-2026/sample_submission.csv
/kaggle/input/competitions/birdclef-2026/taxonomy.csv
/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv
/kaggle/input/competitions/birdclef-2026/train.csv
/kaggle/input/competitions/birdclef-2026/recording_location.txt
/kaggle/input/competitions/birdclef-2026/train_audio/rufgna3/XC252355.ogg
/kaggle/input/competitions/birdclef-2026/train_audio/rufgna3/XC550621.ogg
/kaggle/input/competitions/birdclef-2026/train_audio/rufgna3/XC172127.ogg
/kaggle/input/competitions/birdclef-2026/train_audio/rufgna3/XC499706.ogg
/kaggle/input/competitions/birdclef-2026/train_audio/rufgna3/XC331429.ogg
/kaggle/input/competitions/birdclef-2026/train_audio/rufgna3/XC417588.ogg
/kaggle/input/competitions/birdclef-2026/train_audio/rufgna3/XC80419.ogg
/kaggle/input/competitions/birdclef-2026/train_audio/rufgna3/XC330455.ogg
/kaggle/input/competitions/birdclef-2026/train_audio/rufgna3/XC344738.ogg
/kaggle/input/competitions/birdcl

In [10]:
import os, json
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import timm
from sklearn.preprocessing import MultiLabelBinarizer

BASE = '/kaggle/input/competitions/birdclef-2026'
SR = 32000
CAP = 40
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)

# --- rebuild the same 47-species selection as the baseline ---
train = pd.read_csv(f'{BASE}/train.csv')
ss = pd.read_csv(f'{BASE}/train_soundscapes_labels.csv')

heard = set(s for v in ss['primary_label'].astype(str) for s in v.split(';'))
trainable = set(train['primary_label'].astype(str))
species = sorted(heard & trainable)                       # 47 species
selected = (train[train['primary_label'].astype(str).isin(species)]
            .groupby('primary_label', group_keys=False).head(CAP).reset_index(drop=True))
print("focal clips:", len(selected), "species:", len(species))

mlb = MultiLabelBinarizer(classes=species)
mlb.fit([species])

device: cuda
focal clips: 1445 species: 47


MultiLabelBinarizer(classes=['116570', '22961', '22967', '22973', '23158',
                             '24279', '24321', '25092', '326272', '43435',
                             '47144', '516975', '555146', '65377', '65380',
                             '66971', '67107', '67252', '74113', 'bafcur1',
                             'bufpar', 'bunibi1', 'chacha1', 'chvcon1',
                             'compau', 'compot1', 'fusfly1', 'grekis',
                             'hyamac1', 'limpki', ...])

In [11]:
# --- turn audio into a 3-channel image the CNN expects ---
def spec_image(y):
    target = 5 * SR
    y = np.pad(y, (0, target - len(y))) if len(y) < target else y[:target]
    mel = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=128, fmin=40, fmax=15000)
    mel = librosa.power_to_db(mel, ref=np.max)
    mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-6)   # 0..1
    return mel.astype(np.float32)

# focal training spectrograms
Xf, yf = [], []
for fn, lab in zip(selected['filename'], selected['primary_label'].astype(str)):
    y, _ = librosa.load(f'{BASE}/train_audio/{fn}', sr=SR)
    Xf.append(spec_image(y))
    yf.append([lab])
Xf = np.array(Xf)
Yf = mlb.transform(yf)
print("focal spectrograms:", Xf.shape, "labels:", Yf.shape)

focal spectrograms: (1445, 128, 313) labels: (1445, 47)


In [12]:
# --- soundscape evaluation spectrograms (same cut as before) ---
def t2s(t):
    h, m, s = t.split(':'); return int(h)*3600 + int(m)*60 + int(s)

files = ss['filename'].unique()
Xs, ys, sites = [], [], []
for fname in files:
    y_full, _ = librosa.load(f'{BASE}/train_soundscapes/{fname}', sr=SR)
    site = fname.split('_')[3]
    for _, r in ss[ss['filename'] == fname].iterrows():
        seg = y_full[t2s(r['start'])*SR : t2s(r['end'])*SR]
        Xs.append(spec_image(seg))
        ys.append(r['primary_label'].split(';'))
        sites.append(site)
Xs = np.array(Xs)
Ys = mlb.transform(ys)
print("soundscape spectrograms:", Xs.shape)

soundscape spectrograms: (1478, 128, 313)


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['1491113', '25073', '47158son01', '47158son02', '47158son03', '47158son04', '47158son05', '47158son06', '47158son07', '47158son08', '47158son09', '47158son10', '47158son11', '47158son12', '47158son13', '47158son14', '47158son15', '47158son16', '47158son17', '47158son18', '47158son19', '47158son20', '47158son21', '47158son22', '47158son23', '47158son24', '47158son25', '517063'] will be ignored
  warnings.warn(


In [13]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

class SpecDataset(Dataset):
    def __init__(self, X, Y):
        self.X = X
        self.Y = Y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        x = torch.tensor(self.X[i], dtype=torch.float32).unsqueeze(0)  # (1,128,313)
        x = x.repeat(3, 1, 1)                                          # (3,128,313)
        x = F.interpolate(x.unsqueeze(0), size=(128, 256), mode='bilinear', align_corners=False)[0]
        y = torch.tensor(self.Y[i], dtype=torch.float32)
        return x, y

train_dl = DataLoader(SpecDataset(Xf, Yf), batch_size=32, shuffle=True)
eval_dl  = DataLoader(SpecDataset(Xs, Ys), batch_size=32, shuffle=False)
print("batches:", len(train_dl), "train,", len(eval_dl), "eval")

batches: 46 train, 47 eval


In [14]:
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=47, in_chans=3)
model = model.to(device)

loss_fn = nn.BCEWithLogitsLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 8
for epoch in range(EPOCHS):
    model.train()
    total = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        out = model(xb)
        loss = loss_fn(out, yb)
        loss.backward()
        opt.step()
        total += loss.item()
    print(f"epoch {epoch+1}/{EPOCHS}  loss {total/len(train_dl):.4f}")

epoch 1/8  loss 0.4943
epoch 2/8  loss 0.1262
epoch 3/8  loss 0.0953
epoch 4/8  loss 0.0839
epoch 5/8  loss 0.0741
epoch 6/8  loss 0.0645
epoch 7/8  loss 0.0548
epoch 8/8  loss 0.0465


In [15]:
model.eval()
all_probs = []
with torch.no_grad():
    for xb, _ in eval_dl:
        xb = xb.to(device)
        probs = torch.sigmoid(model(xb))
        all_probs.append(probs.cpu().numpy())

y_prob_eff = np.concatenate(all_probs, axis=0)
print("predictions:", y_prob_eff.shape)

np.save('/kaggle/working/efficientnet_probs.npy', y_prob_eff)
with open('/kaggle/working/efficientnet_meta.json', 'w') as f:
    json.dump({'species': species, 'sites': sites, 'labels': ys}, f)
print("saved")

predictions: (1478, 47)
saved


In [16]:
from sklearn.metrics import roc_auc_score, average_precision_score

pos = Ys.sum(axis=0)
mask = (pos > 0) & (pos < Ys.shape[0])

aucs, aps = [], []
for i in np.where(mask)[0]:
    aucs.append(roc_auc_score(Ys[:, i], y_prob_eff[:, i]))
    aps.append(average_precision_score(Ys[:, i], y_prob_eff[:, i]))

print("species scored:", int(mask.sum()))
print("EfficientNet macro ROC-AUC:", round(np.mean(aucs), 4))
print("EfficientNet macro avg precision:", round(np.mean(aps), 4))

species scored: 47
EfficientNet macro ROC-AUC: 0.5552
EfficientNet macro avg precision: 0.1236


## What happened in this notebook, and why the work moved to Kaggle

This notebook is the working record of extracting Perch embeddings from the BirdCLEF audio. It is left as-is rather than tidied, since its purpose is to document the obstacles and how each was resolved. The clean version of the baseline continues in notebook 09.

The plan was to download the focal training clips locally, embed them with Perch, and keep everything on my own machine. Several problems made that impossible:

**Per-file downloads no longer work.** Both the Kaggle CLI and kagglehub returned 404 errors when fetching individual audio files, and the CLI's folder download was rejected because its flag only accepts single files. Pulling clips one at a time did technically run, but each call re-authenticated from scratch and took around twenty minutes per hundred files, which was far too slow for 1,445 clips.

**Perch could not be installed locally at first.** Perch depends on TensorFlow, and the install failed because the project sat inside a deeply nested OneDrive folder that pushed file paths beyond the Windows 260-character limit. I fixed this by moving the whole project to a short path (C:\dev) and rebuilding the virtual environment in place, since a moved environment keeps its old hardcoded paths and breaks.

**The resolution: embed on Kaggle, analyse locally.** Rather than move tens of gigabytes of audio, I ran the Perch embedding once inside a Kaggle notebook, where the competition data is already mounted and a GPU is available. I then downloaded only the small embedding files. One further detail caught me out here: on Kaggle the competition data mounts under /kaggle/input/competitions/birdclef-2026, one level deeper than the path the documentation implies.

The outcome is two saved embedding files, the focal clips for training and the soundscape segments for evaluation, both produced by the same Perch model. Every later notebook loads these directly, so no audio download or heavy computation is repeated after this point.